## Model Training
<hr>
Processed data used to train and test a TF-IDF Sequential model using TensorFlow.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import hashlib
import pickle
import json
import math
from pathlib import Path
from collections import Counter, defaultdict
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scipy.sparse import load_npz, csr_matrix, coo_matrix
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXTRACTED_DIR = RAW_DIR / "extracted"

X_train = load_npz(PROCESSED_DIR / "lyrics_train.npz")
X_test = load_npz(PROCESSED_DIR / "lyrics_test.npz")

with open(PROCESSED_DIR / "lyrics_metadata.pkl", "rb") as f:
    lyric_metadata = pickle.load(f)

train_ids = lyric_metadata["train_ids"]
test_ids = lyric_metadata["test_ids"]
vocab = lyric_metadata["vocab"]

metadata_df = pd.read_csv(
    PROCESSED_DIR / "track_metadata_lookup.csv",
    dtype={"track_id": str}
)

print("Training matrix:", X_train.shape)
print("Test matrix:", X_test.shape)
print("Training IDs:", len(train_ids))
print("Test IDs:", len(test_ids))
print("Vocabulary size:", len(vocab))
print("Metadata rows:", len(metadata_df))

Training matrix: (210519, 5000)
Test matrix: (27143, 5000)
Training IDs: 210519
Test IDs: 27143
Vocabulary size: 5000
Metadata rows: 237662


In [ ]:
X_train_check = X_train.tocsr(copy=True)
X_test_check = X_test.tocsr(copy=True)

for matrix in (X_train_check, X_test_check):
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    matrix.sort_indices()

train_id_counts = Counter(map(str, train_ids))
test_id_counts = Counter(map(str, test_ids))

train_duplicate_ids = {
    track_id: count
    for track_id, count in train_id_counts.items()
    if count > 1
}
test_duplicate_ids = {
    track_id: count
    for track_id, count in test_id_counts.items()
    if count > 1
}

train_id_set = set(map(str, train_ids))
test_id_set = set(map(str, test_ids))
overlapping_ids = train_id_set & test_id_set

print("DUPLICATE ID CHECK")
print("Duplicate IDs within training:", len(train_duplicate_ids))
print("Duplicate IDs within test:", len(test_duplicate_ids))
print("IDs appearing in both splits:", len(overlapping_ids))

def hash_sparse_rows(matrix):
    signatures = []

    for row_idx in range(matrix.shape[0]):
        start = matrix.indptr[row_idx]
        end = matrix.indptr[row_idx + 1]

        indices = matrix.indices[start:end].astype(np.int64, copy=False)
        values = matrix.data[start:end].astype(np.int64, copy=False)

        digest = hashlib.blake2b(digest_size=16)
        digest.update(indices.tobytes())
        digest.update(values.tobytes())
        signatures.append(digest.hexdigest())

    return signatures


train_hashes = hash_sparse_rows(X_train_check)
test_hashes = hash_sparse_rows(X_test_check)

vector_records = pd.DataFrame({
    "track_id": (
        list(map(str, train_ids)) +
        list(map(str, test_ids))
    ),
    "split": (
        ["train"] * len(train_ids) +
        ["test"] * len(test_ids)
    ),
    "vector_hash": train_hashes + test_hashes
})

distinct_track_vectors = (
    vector_records
    .groupby("vector_hash")
    .agg(
        track_count=("track_id", "nunique"),
        track_ids=("track_id", lambda s: list(dict.fromkeys(s))),
        splits=("split", lambda s: sorted(set(s)))
    )
    .reset_index()
)

duplicate_vector_groups = distinct_track_vectors[
    distinct_track_vectors["track_count"] > 1
].copy()

cross_split_duplicate_vectors = duplicate_vector_groups[
    duplicate_vector_groups["splits"].apply(
        lambda splits: "train" in splits and "test" in splits
    )
].copy()

print("\nIDENTICAL LYRIC VECTOR CHECK")
print("Groups shared by multiple track IDs:", len(duplicate_vector_groups))
print(
    "Groups with identical vectors in train and test:",
    len(cross_split_duplicate_vectors)
)

display(duplicate_vector_groups.head(10))
display(cross_split_duplicate_vectors.head(10))

DUPLICATE ID CHECK
Duplicate IDs within training: 0
Duplicate IDs within test: 0
IDs appearing in both splits: 0

IDENTICAL LYRIC VECTOR CHECK
Groups shared by multiple track IDs: 4626
Groups with identical vectors in train and test: 596


,vector_hash,track_count,track_ids,splits
69,001660e317570e68d705f84c12dba604,2,"[TROYFOI128E0782716, TRXGKNV128E0781D87]",[train]
125,0026f234120daa37973e7b546873be6f,3,"[TRJOVVS128F92D181E, TRNILAA12903CD5665, TRDNY...","[test, train]"
169,0037115aa26d20914a1944a23e46dca1,2,"[TRIPYMD128F9312436, TRPPPQQ128F9312C47]",[train]
214,004540a6cb0b38d2887d3ed70c3a0259,2,"[TRBFSUW12903CD5257, TRENYRF128F931688E]",[train]
298,005e2188393c947a8fa7abf1b5f78882,2,"[TRCBTHQ12903CB89B0, TRPXNJL128F42950FE]",[train]
329,0066c46e97b121a460e23e4304795f35,2,"[TRNDQEC128F424643A, TROCGJX128F422F005]",[train]
339,00692e352887089d12a2ba600c946c6c,2,"[TRXOTQI128F149F63C, TRZHBJQ128F42BA04A]",[train]
342,0069ca49d417d103aabc009322eb4f7a,2,"[TRHFJMC128F935374B, TRQWONP128F427E80C]",[train]
369,00708011ca93f5476e798ba6b6c4195b,2,"[TRRCYBG128F92D3FD1, TRIZGXI128F9311F1C]","[test, train]"
426,00814095146ca18f8c95cb0904c28cdb,2,"[TRYATWN12903CDA434, TRQODJT128F930DD6A]","[test, train]"


,vector_hash,track_count,track_ids,splits
125,0026f234120daa37973e7b546873be6f,3,"[TRJOVVS128F92D181E, TRNILAA12903CD5665, TRDNY...","[test, train]"
369,00708011ca93f5476e798ba6b6c4195b,2,"[TRRCYBG128F92D3FD1, TRIZGXI128F9311F1C]","[test, train]"
426,00814095146ca18f8c95cb0904c28cdb,2,"[TRYATWN12903CDA434, TRQODJT128F930DD6A]","[test, train]"
639,00c0146ab8ab20f7d58b771e63ae64cc,2,"[TRIYEZL128F93203CF, TROLKXV128F9312BFF]","[test, train]"
719,00d6116897ae5beff1b761c120371500,8,"[TRKWFWJ12903CCA83B, TRLKPHD128F42AD04B, TRMAJ...","[test, train]"
1477,01a83cfadaed8193a06c2d268ee0b328,2,"[TRMPWFS128F4287FE5, TRZGCJG128F1482627]","[test, train]"
1974,022958094b1c8df6304acab168206f20,2,"[TRPJQNC128F424C18C, TRDZJBY128F42265A3]","[test, train]"
2292,027cc72bb56d700e17f2b679ef2654c4,2,"[TROUSZN128F92FCC0E, TRQNQRN12903CA4595]","[test, train]"
2455,02a67c82596b7874420f6b9aff51248b,2,"[TRZXFVC128F427076F, TRRWVCM12903CA8272]","[test, train]"
2615,02d2ce1265d7ec281bd5f2ce50fed3b3,2,"[TRXVFAA12903CABFB0, TRFFLVI128F92F217F]","[test, train]"


In [ ]:
train_hash_set = set(train_hashes)

matching_test_positions = [
    i for i, h in enumerate(test_hashes)
    if h in train_hash_set
]

matching_test_ids = [
    str(test_ids[i]) for i in matching_test_positions
]

print("Test tracks with a vector also found in training:",
      len(matching_test_positions))
print("Total test tracks:", len(test_ids))
print("Percentage of test tracks affected:",
      round(100 * len(matching_test_positions) / len(test_ids), 2), "%")

matching_test_ids[:10]

Test tracks with a vector also found in training: 631
Total test tracks: 27143
Percentage of test tracks affected: 2.32 %


['TRAADFO128F92E1E91',
 'TRAAPXO128F424AB85',
 'TRABLNQ128F930E053',
 'TRABQJG128F4267518',
 'TRACEUK128F931DD31',
 'TRAFMPE128F4245349',
 'TRAISQE12903CC45D2',
 'TRAJGIO128F92E84BC',
 'TRAKFYZ128F4279692',
 'TRAKYXD128F933E28B']

In [ ]:
lyric_id_set = set(map(str, train_ids)) | set(map(str, test_ids))

tag_records = []
found_json_ids = set()
json_files_checked = 0

for json_path in EXTRACTED_DIR.rglob("*.json"):
    json_files_checked += 1
    track_id = json_path.stem

    if track_id not in lyric_id_set:
        continue

    found_json_ids.add(track_id)

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        raw_tags = data.get("tags", [])

        if not isinstance(raw_tags, list):
            continue

        for item in raw_tags:
            if not isinstance(item, (list, tuple)) or len(item) < 1:
                continue

            tag = str(item[0]).strip().lower()

            if not tag:
                continue

            try:
                weight = float(item[1]) if len(item) > 1 else 1.0
            except (TypeError, ValueError):
                weight = 1.0

            tag_records.append({
                "track_id": track_id,
                "tag": tag,
                "weight": weight
            })

    except (OSError, json.JSONDecodeError) as exc:
        print(f"Could not read {json_path}: {exc}")

tags_df = pd.DataFrame(
    tag_records,
    columns=["track_id", "tag", "weight"]
)

if not tags_df.empty:
    tags_df = (
        tags_df
        .groupby(["track_id", "tag"], as_index=False)["weight"]
        .max()
    )

print("JSON files checked:", json_files_checked)
print("Lyric track IDs:", len(lyric_id_set))
print("Matching JSON files found:", len(found_json_ids))
print("Tracks with at least one usable tag:",
      tags_df["track_id"].nunique())
print("Unique tag labels:", tags_df["tag"].nunique())

display(tags_df.head())

JSON files checked: 943334
Lyric track IDs: 237662
Matching JSON files found: 231009
Tracks with at least one usable tag: 173730
Unique tag labels: 371439


,track_id,tag,weight
0,TRAAABD128F429CF47,1960's,5.0
1,TRAAABD128F429CF47,1960s,11.0
2,TRAAABD128F429CF47,1969,23.0
3,TRAAABD128F429CF47,60's,5.0
4,TRAAABD128F429CF47,60s,100.0


In [ ]:
tagged_track_ids = set(tags_df["track_id"].astype(str).unique())

train_id_set = set(map(str, train_ids))
test_id_set = set(map(str, test_ids))

coverage = pd.DataFrame([
    {
        "split": "train",
        "total_tracks": len(train_id_set),
        "tagged_tracks": len(train_id_set & tagged_track_ids),
    },
    {
        "split": "test",
        "total_tracks": len(test_id_set),
        "tagged_tracks": len(test_id_set & tagged_track_ids),
    },
])

coverage["untagged_tracks"] = (
    coverage["total_tracks"] - coverage["tagged_tracks"]
)

coverage["tagged_percentage"] = (
    100 * coverage["tagged_tracks"] / coverage["total_tracks"]
).round(2)

display(coverage)

,split,total_tracks,tagged_tracks,untagged_tracks,tagged_percentage
0,train,210519,153626,56893,72.97
1,test,27143,20104,7039,74.07


In [ ]:
tag_frequency = (
    tags_df.groupby("tag")["track_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("track_count")
    .reset_index()
)

print("Number of unique tags:", len(tag_frequency))

print("\nTop 30 most frequent tags:")
display(tag_frequency.head(30))

print("\nTag frequency distribution:")
display(
    tag_frequency["track_count"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
)

Number of unique tags: 371439

Top 30 most frequent tags:


,tag,track_count
0,rock,55991
1,pop,35672
2,alternative,32021
3,indie,26270
4,female vocalists,21534
5,favorites,21430
6,alternative rock,18897
7,love,18038
8,metal,17744
9,00s,16566



Tag frequency distribution:


count    371439.000000
mean         10.582588
std         203.583229
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
90%          11.000000
95%          23.000000
max       55991.000000
Name: track_count, dtype: float64

In [ ]:
train_ids_str = set(map(str, train_ids))

train_tags_df = tags_df[
    tags_df["track_id"].astype(str).isin(train_ids_str)
].copy()

train_tag_frequency = (
    train_tags_df.groupby("tag")["track_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("train_track_count")
    .reset_index()
)

thresholds = [100, 250, 500, 1000, 2000]
threshold_summary = pd.DataFrame([
    {
        "minimum_training_tracks": threshold,
        "number_of_tags": int(
            (train_tag_frequency["train_track_count"] >= threshold).sum()
        )
    }
    for threshold in thresholds
])

print("Number of eligible tags at each threshold:")
display(threshold_summary)

print("\nTop 30 tags by training frequency:")
display(train_tag_frequency.head(30))

Number of eligible tags at each threshold:


,minimum_training_tracks,number_of_tags
0,100,3643
1,250,1382
2,500,664
3,1000,338
4,2000,169



Top 30 tags by training frequency:


,tag,train_track_count
0,rock,50498
1,pop,31923
2,alternative,29019
3,indie,23483
4,favorites,19426
5,female vocalists,19178
6,alternative rock,17100
7,love,16269
8,metal,15561
9,singer-songwriter,14939


In [ ]:
MIN_TAG_FREQUENCY = 500
EXCLUDED_TAGS = {
    "favorite",
    "favorites",
    "favourites",
    "seen live",
    "owned",
    "to-buy",
    "to buy",
    "lastfm",
    "last.fm",
    "my music",
    "my favorites",
    "playlist",
    "spotify",
}

selected_tag_frequency = train_tag_frequency[
    (train_tag_frequency["train_track_count"] >= MIN_TAG_FREQUENCY)
    & (~train_tag_frequency["tag"].isin(EXCLUDED_TAGS))
].copy()

selected_tags = sorted(selected_tag_frequency["tag"].unique())
selected_tag_set = set(selected_tags)

print("Minimum training frequency:", MIN_TAG_FREQUENCY)
print("Tags before exclusions meeting threshold:",
      int((train_tag_frequency["train_track_count"] >= MIN_TAG_FREQUENCY).sum()))
print("Excluded personal/platform tags:",
      len(set(train_tag_frequency["tag"]) & EXCLUDED_TAGS))
print("Final selected tag count:", len(selected_tags))

print("\nSelected tag frequency summary:")
display(selected_tag_frequency["train_track_count"].describe())

print("\nTop 30 selected tags:")
display(selected_tag_frequency.head(30))

Minimum training frequency: 500
Tags before exclusions meeting threshold: 664
Excluded personal/platform tags: 13
Final selected tag count: 658

Selected tag frequency summary:


count      658.000000
mean      2155.679331
std       3622.846682
min        501.000000
25%        655.250000
50%       1007.000000
75%       2006.000000
max      50498.000000
Name: train_track_count, dtype: float64


Top 30 selected tags:


,tag,train_track_count
0,rock,50498
1,pop,31923
2,alternative,29019
3,indie,23483
5,female vocalists,19178
6,alternative rock,17100
7,love,16269
8,metal,15561
9,singer-songwriter,14939
10,00s,14886


The Last.fm datasets from the Million Song Dataset already came split into training and testing sets, so it mainly just needs matching to the vocabulary.

In [ ]:
selected_tag_rows = tags_df[
    tags_df["tag"].isin(selected_tag_set)
].copy()

train_selected_rows = selected_tag_rows[
    selected_tag_rows["track_id"].isin(train_id_set)
]

test_selected_rows = selected_tag_rows[
    selected_tag_rows["track_id"].isin(test_id_set)
]

train_tracks_with_selected_tags = (
    train_selected_rows["track_id"].nunique()
)
test_tracks_with_selected_tags = (
    test_selected_rows["track_id"].nunique()
)

selected_coverage = pd.DataFrame([
    {
        "split": "train",
        "total_tracks": len(train_id_set),
        "tracks_with_selected_tags": train_tracks_with_selected_tags,
    },
    {
        "split": "test",
        "total_tracks": len(test_id_set),
        "tracks_with_selected_tags": test_tracks_with_selected_tags,
    },
])

selected_coverage["percentage"] = (
    100 * selected_coverage["tracks_with_selected_tags"]
    / selected_coverage["total_tracks"]
).round(2)
display(selected_coverage)

,split,total_tracks,tracks_with_selected_tags,percentage
0,train,210519,140348,66.67
1,test,27143,18443,67.95


In [ ]:
train_ids = np.asarray(train_ids, dtype=str)
test_ids = np.asarray(test_ids, dtype=str)

tags_df["track_id"] = tags_df["track_id"].astype(str)
tags_df["tag"] = tags_df["tag"].astype(str)

selected_tags = sorted(selected_tag_set)
tag_to_index = {tag: i for i, tag in enumerate(selected_tags)}

def build_label_matrix(track_ids, tags_df, tag_to_index):
    """Build a sparse multi-hot label matrix aligned to track_ids."""
    track_to_index = {
        track_id: i for i, track_id in enumerate(track_ids)
    }

    relevant_tags = tags_df[
        tags_df["track_id"].isin(track_to_index)
        & tags_df["tag"].isin(tag_to_index)
    ][["track_id", "tag"]].drop_duplicates()

    row_indices = relevant_tags["track_id"].map(
        track_to_index
    ).to_numpy()

    col_indices = relevant_tags["tag"].map(
        tag_to_index
    ).to_numpy()

    labels = coo_matrix(
        (
            np.ones(len(relevant_tags), dtype=np.float32),
            (row_indices, col_indices)
        ),
        shape=(len(track_ids), len(tag_to_index)),
        dtype=np.float32
    ).tocsr()

    labels.data[:] = 1.0
    labels.eliminate_zeros()

    return labels

Y_train_all = build_label_matrix(
    train_ids, tags_df, tag_to_index
)

Y_test_all = build_label_matrix(
    test_ids, tags_df, tag_to_index
)

print("Training label matrix:", Y_train_all.shape)
print("Test label matrix:", Y_test_all.shape)
print("Selected tags:", len(selected_tags))
print(
    "Training tracks with labels:",
    int(np.count_nonzero(Y_train_all.getnnz(axis=1)))
)
print(
    "Test tracks with labels:",
    int(np.count_nonzero(Y_test_all.getnnz(axis=1)))
)

Training label matrix: (210519, 658)
Test label matrix: (27143, 658)
Selected tags: 658
Training tracks with labels: 140348
Test tracks with labels: 18443


In [12]:
train_mask = Y_train_all.getnnz(axis=1) > 0
test_mask = Y_test_all.getnnz(axis=1) > 0

X_train_labeled = X_train[train_mask].tocsr()
X_test_labeled = X_test[test_mask].tocsr()

Y_train = Y_train_all[train_mask].tocsr()
Y_test = Y_test_all[test_mask].tocsr()

train_ids_labeled = train_ids[train_mask]
test_ids_labeled = test_ids[test_mask]

print("Labeled training features:", X_train_labeled.shape)
print("Labeled training targets:", Y_train.shape)
print("Labeled test features:", X_test_labeled.shape)
print("Labeled test targets:", Y_test.shape)

print(
    f"Training coverage: {train_mask.mean():.2%}"
)
print(
    f"Test coverage: {test_mask.mean():.2%}"
)

Labeled training features: (140348, 5000)
Labeled training targets: (140348, 658)
Labeled test features: (18443, 5000)
Labeled test targets: (18443, 658)
Training coverage: 66.67%
Test coverage: 67.95%


<b>TF-IDF feature extraction and preprocessing</b>

In [13]:
tfidf = TfidfTransformer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_labeled)
X_test_tfidf = tfidf.transform(X_test_labeled)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)
print("Training matrix density:", f"{X_train_tfidf.nnz / np.prod(X_train_tfidf.shape):.4%}")

TF-IDF training shape: (140348, 5000)
TF-IDF test shape: (18443, 5000)
Training matrix density: 1.6138%


<b>Using TensorFlow to fit the data to the statistical preprocessing algorithm</b>

In [ ]:
X_train_tfidf = X_train_tfidf.astype(np.float32).tocsr()
X_test_tfidf = X_test_tfidf.astype(np.float32).tocsr()

all_train_indices = np.arange(X_train_tfidf.shape[0])

fit_indices, val_indices = train_test_split(
    all_train_indices,
    test_size=0.10,
    random_state=42
)

print("Model-training tracks:", len(fit_indices))
print("Validation tracks:", len(val_indices))
print("Held-out test tracks:", X_test_tfidf.shape[0])
print("TensorFlow version:", tf.__version__)

Model-training tracks: 126313
Validation tracks: 14035
Held-out test tracks: 18443
TensorFlow version: 2.16.2


In [15]:
BATCH_SIZE = 256
N_FEATURES = X_train_tfidf.shape[1]
N_TAGS = Y_train.shape[1]

def make_dataset(X, Y, indices, batch_size=256, shuffle=False):
    def generator():
        order = np.array(indices, copy=True)

        if shuffle:
            np.random.shuffle(order)

        for start in range(0, len(order), batch_size):
            batch_indices = order[start:start + batch_size]

            X_batch = X[batch_indices].toarray().astype(np.float32)
            Y_batch = Y[batch_indices].toarray().astype(np.float32)

            yield X_batch, Y_batch

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None, N_FEATURES), dtype=tf.float32),
            tf.TensorSpec(shape=(None, N_TAGS), dtype=tf.float32),
        ),
    )

    return dataset.prefetch(1)


train_ds = make_dataset(
    X_train_tfidf, Y_train, fit_indices,
    batch_size=BATCH_SIZE, shuffle=True
)

val_ds = make_dataset(
    X_train_tfidf, Y_train, val_indices,
    batch_size=BATCH_SIZE
)

test_indices = np.arange(X_test_tfidf.shape[0])

test_ds = make_dataset(
    X_test_tfidf, Y_test, test_indices,
    batch_size=BATCH_SIZE
)

steps_per_epoch = math.ceil(len(fit_indices) / BATCH_SIZE)
validation_steps = math.ceil(len(val_indices) / BATCH_SIZE)
test_steps = math.ceil(len(test_indices) / BATCH_SIZE)

print("Training batches per epoch:", steps_per_epoch)
print("Validation batches:", validation_steps)
print("Test batches:", test_steps)

Training batches per epoch: 494
Validation batches: 55
Test batches: 73


In [ ]:
train_ds = make_dataset(
    X_train_tfidf,
    Y_train,
    fit_indices,
    batch_size=BATCH_SIZE,
    shuffle=True,
).repeat()

val_ds = make_dataset(
    X_train_tfidf,
    Y_train,
    val_indices,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Training batches per epoch:", steps_per_epoch)
print("Validation batches:", validation_steps)

Training batches per epoch: 494
Validation batches: 55


<b>Initializing the Sequential model</b><br>
The sequential model was chosen since the mapping from TF-IDF word frequencies to musical tags requires a straightforward, linear data flow.

In [17]:
model = Sequential([
    Input(shape=(N_FEATURES,)),
    Dense(512, activation="relu"),
    Dropout(0.35),
    Dense(256, activation="relu"),
    Dropout(0.25),
    Dense(N_TAGS, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     2,560,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 658)            │       169,106 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,860,946 (10.91 MB)

 Trainable params: 2,860,946 (10.91 MB)

 Non-trainable params: 0 (0.00 B)

<b>Fitting the Sequential model</b><br>
The model stopped early at epoch 10, and determined that the epoch leading to the best results was 7.

In [18]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    epochs=15,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/15


c:\Users\Mina\Documents\CSU_MSDS\Found of AI\TensorFlow\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


494/494 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 0.0906 - val_loss: 0.0674 - learning_rate: 0.0010
Epoch 2/15
494/494 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0658 - val_loss: 0.0642 - learning_rate: 0.0010
Epoch 3/15
494/494 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0635 - val_loss: 0.0632 - learning_rate: 0.0010
Epoch 4/15
494/494 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0624 - val_loss: 0.0626 - learning_rate: 0.0010
Epoch 5/15
494/494 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0615 - val_loss: 0.0623 - learning_rate: 0.0010
Epoch 6/15
494/494 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0608 - val_loss: 0.0622 - learning_rate: 0.0010
Epoch 7/15
494/494 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0601 - val_loss: 0.0620 - learning_rate: 0.0010
Epoch 8/15
492/494 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0593
Epoch 8: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
494/494 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0593 - val_loss: 0.0621 - learning_rate: 0.0010

In [19]:
test_loss = model.evaluate(
    test_ds,
    steps=test_steps,
    verbose=1
)

print(f"Test binary cross-entropy: {test_loss:.5f}")

73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0544
Test binary cross-entropy: 0.05444


It does seem to struggle a bit to predict enough tags per track, as you can see the model predicts 0.59 tags on average, when the actual tags per song averages at 8.59.

In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

y_prob = model.predict(test_ds, steps=test_steps, verbose=1)

y_true = Y_test.toarray().astype(np.int8)

y_pred = (y_prob >= 0.5).astype(np.int8)

print("Test metrics at threshold 0.5")
print(f"Micro precision: {precision_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
print(f"Micro recall:    {recall_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
print(f"Micro F1:        {f1_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
print(f"Macro F1:        {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

print("Average predicted tags per track:", round(y_pred.sum(axis=1).mean(), 2))
print("Average actual tags per track:", round(y_true.sum(axis=1).mean(), 2))

73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Test metrics at threshold 0.5
Micro precision: 0.5509
Micro recall:    0.0381
Micro F1:        0.0713
Macro F1:        0.0167
Average predicted tags per track: 0.59
Average actual tags per track: 8.59


In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd

val_prob = model.predict(
    val_ds,
    steps=validation_steps,
    verbose=1
)

y_val = Y_train[val_indices].toarray().astype(np.int8)

threshold_results = []

for threshold in [0.01, 0.02, 0.03, 0.05, 0.075, 0.10,
                  0.15, 0.20, 0.30, 0.40, 0.50]:
    val_pred = (val_prob >= threshold).astype(np.int8)

    threshold_results.append({
        "threshold": threshold,
        "micro_precision": precision_score(
            y_val, val_pred, average="micro", zero_division=0
        ),
        "micro_recall": recall_score(
            y_val, val_pred, average="micro", zero_division=0
        ),
        "micro_f1": f1_score(
            y_val, val_pred, average="micro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_val, val_pred, average="macro", zero_division=0
        ),
        "avg_predicted_tags": val_pred.sum(axis=1).mean(),
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))

55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


,threshold,micro_precision,micro_recall,micro_f1,macro_f1,avg_predicted_tags
0,0.010,0.0448,0.8263,0.0850,0.0647,188.1957
1,0.020,0.0681,0.7045,0.1241,0.0824,105.6129
2,0.030,0.0880,0.6177,0.1541,0.0894,71.6086
3,0.050,0.1239,0.4986,0.1985,0.0925,41.0558
4,0.075,0.1646,0.4007,0.2333,0.0897,24.8448
5,0.100,0.2014,0.3316,0.2506,0.0826,16.8029
6,0.150,0.2677,0.2363,0.2511,0.0682,9.0081
7,0.200,0.3280,0.1769,0.2299,0.0554,5.5053
8,0.300,0.4237,0.1064,0.1700,0.0364,2.5612
9,0.400,0.4991,0.0635,0.1126,0.0255,1.2974


The best threshold value for our model to predicte as close to the actual average of tags per track (8.59) is 0.15, which predicts 9.01 tags on average.

In [22]:
best_row = threshold_df.loc[threshold_df["micro_f1"].idxmax()]
best_threshold = float(best_row["threshold"])

print("Best validation threshold:", best_threshold)
print("Validation micro F1:", round(best_row["micro_f1"], 4))
print("Validation precision:", round(best_row["micro_precision"], 4))
print("Validation recall:", round(best_row["micro_recall"], 4))
print("Average predicted tags per track:",
      round(best_row["avg_predicted_tags"], 2))

Best validation threshold: 0.15
Validation micro F1: 0.2511
Validation precision: 0.2677
Validation recall: 0.2363
Average predicted tags per track: 9.01


Evaluating a held-out test set using this 0.15 threshold results in an average of 8.74 predicted tags per track.

In [23]:
test_pred = (y_prob >= best_threshold).astype(np.int8)

print(f"Test results at threshold {best_threshold:.3f}")
print(f"Micro precision: {precision_score(y_true, test_pred, average='micro', zero_division=0):.4f}")
print(f"Micro recall:    {recall_score(y_true, test_pred, average='micro', zero_division=0):.4f}")
print(f"Micro F1:        {f1_score(y_true, test_pred, average='micro', zero_division=0):.4f}")
print(f"Macro F1:        {f1_score(y_true, test_pred, average='macro', zero_division=0):.4f}")
print(f"Average predicted tags per track: {test_pred.sum(axis=1).mean():.2f}")
print(f"Average actual tags per track:    {y_true.sum(axis=1).mean():.2f}")

Test results at threshold 0.150
Micro precision: 0.2396
Micro recall:    0.2437
Micro F1:        0.2416
Macro F1:        0.0616
Average predicted tags per track: 8.74
Average actual tags per track:    8.59


In [ ]:
tag_frequencies = np.asarray(
    Y_train[fit_indices].sum(axis=0)
).ravel()

ranked_tag_indices = np.argsort(tag_frequencies)[::-1]
y_val = Y_train[val_indices].toarray().astype(np.int8)

baseline_results = []

for n_tags in [5, 8, 10, 15, 20, 30]:
    selected_indices = ranked_tag_indices[:n_tags]

    val_pred = np.zeros_like(y_val)
    val_pred[:, selected_indices] = 1

    baseline_results.append({
        "tags_predicted_per_track": n_tags,
        "micro_precision": precision_score(
            y_val, val_pred, average="micro", zero_division=0
        ),
        "micro_recall": recall_score(
            y_val, val_pred, average="micro", zero_division=0
        ),
        "micro_f1": f1_score(
            y_val, val_pred, average="micro", zero_division=0
        ),
    })

baseline_df = pd.DataFrame(baseline_results)
display(baseline_df.round(4))

,tags_predicted_per_track,micro_precision,micro_recall,micro_f1
0,5,0.2222,0.1089,0.1462
1,8,0.1827,0.1433,0.1606
2,10,0.1678,0.1645,0.1661
3,15,0.1428,0.2100,0.1700
4,20,0.1273,0.2495,0.1686
5,30,0.1076,0.3162,0.1605


In [ ]:
n_tags = 15
selected_indices = ranked_tag_indices[:n_tags]

y_test = Y_test.toarray().astype(np.int8)

baseline_test_pred = np.zeros_like(y_test)
baseline_test_pred[:, selected_indices] = 1

print("Baseline test results (top 15 tags)")
print(
    "Micro precision:",
    round(precision_score(
        y_test, baseline_test_pred,
        average="micro", zero_division=0
    ), 4)
)
print(
    "Micro recall:   ",
    round(recall_score(
        y_test, baseline_test_pred,
        average="micro", zero_division=0
    ), 4)
)
print(
    "Micro F1:       ",
    round(f1_score(
        y_test, baseline_test_pred,
        average="micro", zero_division=0
    ), 4)
)

Baseline test results (top 15 tags)
Micro precision: 0.1183
Micro recall:    0.2066
Micro F1:        0.1504


In [26]:
track_metadata_lookup = pd.read_csv(
    PROCESSED_DIR / "track_metadata_lookup.csv"
)

print("Metadata shape:", track_metadata_lookup.shape)
print(track_metadata_lookup.head())
print(track_metadata_lookup["split"].value_counts())

Metadata shape: (237662, 6)
             track_id split                                title  \
0  TRAABRX12903CC4816  test  What Child Is This? (Album Version)   
1  TRAADFO128F92E1E91  test                         In The House   
2  TRAADQW128F427CE68  test                             Prophecy   
3  TRAADRX12903D0EFE8  test                   Sulla buona strada   
4  TRAAEJQ128F92C484E  test                     Four Letter Word   

       artist_name                   release  year  
0         Plus One                 Christmas  2002  
1          Asher D             Still Kickin'     0  
2            Xymox                Headclouds  1993  
3   Riccardo Fogli  Sanremo è sempre Sanremo  1976  
4  Raheem Devaughn    Love Behind The Melody  2008  
split
train    210519
test      27143
Name: count, dtype: int64


In [ ]:
test_metadata = track_metadata_lookup[
    track_metadata_lookup["split"] == "test"
].copy()

test_metadata = (
    test_metadata.set_index("track_id")
    .reindex(test_ids)
    .reset_index()
)

rng = np.random.default_rng(42)
eligible = np.flatnonzero(y_true.sum(axis=1) > 0)
sample_indices = rng.choice(
    eligible,
    size=min(5, len(eligible)),
    replace=False
)

for i in sample_indices:
    top_indices = np.argsort(y_prob[i])[::-1][:10]
    predicted_tags = [
        (selected_tags[j], round(float(y_prob[i, j]), 3))
        for j in top_indices
    ]

    actual_tags = [
        selected_tags[j]
        for j in np.flatnonzero(y_true[i])
    ]

    predicted_at_threshold = {
        selected_tags[j]
        for j in np.flatnonzero(y_prob[i] >= 0.15)
    }
    actual_set = set(actual_tags)

    metadata = test_metadata.iloc[i]

    print("=" * 70)
    print(f"Track:  {metadata.get('title', 'Unknown')}")
    print(f"Artist: {metadata.get('artist_name', 'Unknown')}")
    print("Top 10 predicted tags (tag, score):")
    print(predicted_tags)
    print("Known tags:")
    print(actual_tags)
    print("Correct predicted tags:")
    print(sorted(predicted_at_threshold & actual_set))

Track:  Not Of This God
Artist: Slayer
Top 10 predicted tags (tag, score):
[('punk', 0.249), ('rock', 0.189), ('punk rock', 0.147), ('alternative', 0.128), ('political', 0.109), ('hardcore punk', 0.109), ('hardcore', 0.105), ('indie', 0.099), ('folk', 0.064), ('alternative rock', 0.057)]
Known tags:
['tantotempotaste']
Correct predicted tags:
[]
Track:  ...Then I Defy You_ Stars (LP Version)
Artist: The Receiving End Of Sirens
Top 10 predicted tags (tag, score):
[('pop', 0.572), ('rnb', 0.43), ('soul', 0.407), ('dance', 0.346), ('love', 0.302), ('female vocalists', 0.3), ('rock', 0.248), ('sexy', 0.184), ('love songs', 0.181), ('r&b', 0.174)]
Known tags:
['00s', 'british', 'britpop', 'disco', 'female vocalist', 'female vocalists', 'guilty pleasures', 'pop']
Correct predicted tags:
['00s', 'female vocalist', 'female vocalists', 'pop']
Track:  Sunday Driver
Artist: Buck 65
Top 10 predicted tags (tag, score):
[('indie', 0.532), ('alternative', 0.439), ('rock', 0.426), ('indie rock', 0.316

In [ ]:
sample_titles = [
    "Not Of This God",
    "...Then I Defy You_ Stars (LP Version)",
    "Sunday Driver",
    "Horrorscope",
    "Irish Blood_ English Heart",
]

sample_artists = [
    "Slayer",
    "The Receiving End Of Sirens",
    "Buck 65",
    "PRIMAL FEAR",
    "Morrissey",
]

rows = []

for title, artist in zip(sample_titles, sample_artists):
    matches = test_metadata[
        (test_metadata["title"] == title)
        & (test_metadata["artist_name"] == artist)
    ]

    if matches.empty:
        print(f"Not found in metadata: {title} — {artist}")
        continue

    track_id = matches.iloc[0]["track_id"]
    positions = [
        j for j, tid in enumerate(test_ids)
        if tid == track_id
    ]

    if not positions:
        print(f"Track ID not found in test_ids: {track_id}")
        continue

    i = positions[0]
    top_indices = np.argsort(y_prob[i])[::-1][:10]

    for rank, j in enumerate(top_indices, start=1):
        rows.append({
            "title": title,
            "artist": artist,
            "rank": rank,
            "predicted_tag": selected_tags[j],
            "score": round(float(y_prob[i, j]), 3),
            "known_label": bool(y_true[i, j]),
        })

prediction_inspection = pd.DataFrame(rows)
display(prediction_inspection)

,title,artist,rank,predicted_tag,score,known_label
0,Not Of This God,Slayer,1,punk,0.249,False
1,Not Of This God,Slayer,2,rock,0.189,False
2,Not Of This God,Slayer,3,punk rock,0.147,False
3,Not Of This God,Slayer,4,alternative,0.128,False
4,Not Of This God,Slayer,5,political,0.109,False
5,Not Of This God,Slayer,6,hardcore punk,0.109,False
6,Not Of This God,Slayer,7,hardcore,0.105,False
7,Not Of This God,Slayer,8,indie,0.099,False
8,Not Of This God,Slayer,9,folk,0.064,False
9,Not Of This God,Slayer,10,alternative rock,0.057,False


The tags assigned to Primal Fear's "Horrorscope" are slightly suspicisous, so we investigate it further.

In [ ]:
matches = test_metadata[
    (test_metadata["title"] == "Horrorscope")
    & (test_metadata["artist_name"].str.upper() == "PRIMAL FEAR")
]

if matches.empty:
    print("Track not found in test metadata.")
else:
    track_id = matches.iloc[0]["track_id"]
    positions = [
        i for i, tid in enumerate(test_ids)
        if tid == track_id
    ]

    print(matches.to_string(index=False))
    print("Track ID:", track_id)

    if positions:
        i = positions[0]
        print("Nonzero lyric features:", X_test[i].nnz)
        print("Total word counts:", X_test[i].sum())
        print("Known selected tags:", [
            selected_tags[j]
            for j in np.flatnonzero(y_true[i])
        ])

          track_id split       title artist_name     release  year
TRBQBCF128F424C0A6  test Horrorscope PRIMAL FEAR Horrorscope  1999
Track ID: TRBQBCF128F424C0A6
Nonzero lyric features: 5
Total word counts: 6
Known selected tags: ['acoustic', 'animals', 'ballad', 'beautiful', 'canadian', 'downtempo', 'fav', 'folk', 'folk rock', 'great songs', 'indie', 'love at first listen', 'poprock', 'rock', 'singer-songwriter']


In [ ]:
train_unique_terms = np.diff(X_train.indptr)
test_unique_terms = np.diff(X_test.indptr)

train_word_counts = np.asarray(X_train.sum(axis=1)).ravel()
test_word_counts = np.asarray(X_test.sum(axis=1)).ravel()

sparsity_summary = pd.DataFrame({
    "split": ["Train", "Test"],
    "tracks": [X_train.shape[0], X_test.shape[0]],
    "under_5_unique_terms": [
        np.sum(train_unique_terms < 5),
        np.sum(test_unique_terms < 5),
    ],
    "under_10_unique_terms": [
        np.sum(train_unique_terms < 10),
        np.sum(test_unique_terms < 10),
    ],
    "under_20_total_word_counts": [
        np.sum(train_word_counts < 20),
        np.sum(test_word_counts < 20),
    ],
})

display(sparsity_summary)

,split,tracks,under_5_unique_terms,under_10_unique_terms,under_20_total_word_counts
0,Train,210519,745,2522,3042
1,Test,27143,99,361,429


In [ ]:
labeled_word_counts = np.asarray(
    X_test_labeled.sum(axis=1)
).ravel()

print("Labeled test feature rows:", X_test_labeled.shape[0])
print("Prediction rows:", y_prob.shape[0])
print("True-label rows:", y_true.shape[0])
print("Word-count rows:", len(labeled_word_counts))

assert len(labeled_word_counts) == y_prob.shape[0] == y_true.shape[0]

non_sparse_mask = labeled_word_counts >= 20

non_sparse_pred = (
    y_prob[non_sparse_mask] >= 0.15
).astype(np.int8)

non_sparse_true = y_true[non_sparse_mask]

precision = precision_score(
    non_sparse_true, non_sparse_pred, average="micro", zero_division=0
)
recall = recall_score(
    non_sparse_true, non_sparse_pred, average="micro", zero_division=0
)
f1 = f1_score(
    non_sparse_true, non_sparse_pred, average="micro", zero_division=0
)

print(f"\nNon-sparse test tracks: {non_sparse_mask.sum():,}")
print(f"Excluded sparse tracks: {(~non_sparse_mask).sum():,}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Micro F1:  {f1:.4f}")


Labeled test feature rows: 18443
Prediction rows: 18443
True-label rows: 18443
Word-count rows: 18443

Non-sparse test tracks: 18,175
Excluded sparse tracks: 268
Precision: 0.2398
Recall:    0.2441
Micro F1:  0.2419


In [ ]:
print("Y_train:", Y_train.shape)
print("Y_test:", Y_test.shape)
print("y_prob:", y_prob.shape)
print("y_true:", y_true.shape)

for name in ["selected_tags", "tag_names", "selected_tag_names", "tags"]:
    if name in globals():
        value = globals()[name]
        print(f"\n{name}: type={type(value).__name__}, length={len(value)}")
        print("First 10:", list(value)[:10])

Y_train: (140348, 658)
Y_test: (18443, 658)
y_prob: (18443, 658)
y_true: (18443, 658)

selected_tags: type=list, length=658
First 10: ['00s', '1', '131', '1960s', '1970s', '1980s', '1990s', '1994', '1995', '1996']


In [ ]:
y_train_dense = Y_train.toarray() if hasattr(Y_train, "toarray") else np.asarray(Y_train)
y_test_dense = y_true

tag_frequency = y_train_dense.sum(axis=0).astype(int)

y_pred = (y_prob >= 0.15).astype(np.int8)

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_dense,
    y_pred,
    average=None,
    zero_division=0
)

tag_results = pd.DataFrame({
    "tag": list(selected_tags),
    "train_frequency": tag_frequency,
    "test_support": support,
    "precision": precision,
    "recall": recall,
    "f1": f1
})

tag_results = tag_results.sort_values(
    "train_frequency", ascending=False
).reset_index(drop=True)

display(tag_results.head(20))

,tag,train_frequency,test_support,precision,recall,f1
0,rock,50498,5493,0.331049,0.958493,0.492125
1,pop,31923,3749,0.275217,0.895172,0.421000
2,alternative,29019,3002,0.242906,0.829780,0.375801
3,indie,23483,2787,0.299836,0.787944,0.434378
4,female vocalists,19178,2356,0.206965,0.617997,0.310084
5,alternative rock,17100,1797,0.190608,0.652755,0.295057
6,love,16269,1769,0.179891,0.580554,0.274672
7,metal,15561,2183,0.360071,0.744388,0.485364
8,singer-songwriter,14939,1353,0.156147,0.455285,0.232541
9,00s,14886,1680,0.165648,0.419643,0.237534


In [34]:
frequency_bins = [0, 999, 2499, 4999, 9999, np.inf]
frequency_labels = [
    "500–999",
    "1,000–2,499",
    "2,500–4,999",
    "5,000–9,999",
    "10,000+"
]

tag_results["frequency_group"] = pd.cut(
    tag_results["train_frequency"],
    bins=frequency_bins,
    labels=frequency_labels,
    include_lowest=True
)

group_results = (
    tag_results.groupby("frequency_group", observed=False)
    .agg(
        number_of_tags=("tag", "count"),
        mean_train_frequency=("train_frequency", "mean"),
        mean_test_support=("test_support", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        median_f1=("f1", "median")
    )
    .round(4)
)

display(group_results)

,number_of_tags,mean_train_frequency,mean_test_support,mean_precision,mean_recall,mean_f1,median_f1
frequency_group,,,,,,,
500–999,326,679.9847,76.3344,0.0523,0.0357,0.0307,0.0000
"1,000–2,499",188,1481.7074,169.2979,0.0921,0.0533,0.0511,0.0000
"2,500–4,999",82,3377.3049,383.6463,0.1342,0.1077,0.0981,0.0127
"5,000–9,999",38,6483.7632,719.0526,0.1802,0.1992,0.1705,0.1209
"10,000+",24,16453.2917,1788.9167,0.1931,0.4735,0.2676,0.2412


In [ ]:
tag_results["predicted_count"] = y_pred.sum(axis=0)
tag_results["true_positives"] = (
    y_pred.astype(bool) & y_test_dense.astype(bool)
).sum(axis=0)

tag_results["false_positives"] = (
    y_pred.astype(bool) & ~y_test_dense.astype(bool)
).sum(axis=0)

zero_f1_tags = (
    tag_results[tag_results["f1"] == 0]
    .sort_values("train_frequency", ascending=False)
)

display(
    zero_f1_tags[
        [
            "tag",
            "train_frequency",
            "test_support",
            "predicted_count",
            "true_positives",
            "false_positives",
            "precision",
            "recall",
            "f1",
        ]
    ].head(25)
)

print(f"Tags with zero F1: {(tag_results['f1'] == 0).sum()} / {len(tag_results)}")
print(f"Tags never predicted: {(tag_results['predicted_count'] == 0).sum()}")

,tag,train_frequency,test_support,predicted_count,true_positives,false_positives,precision,recall,f1
40,favourite,6275,604,0,0,0,0.0,0.0,0.0
47,favorite songs,5981,588,0,0,0,0.0,0.0,0.0
59,loved,5297,589,0,0,0,0.0,0.0,0.0
62,heard on pandora,4983,568,0,0,0,0.0,0.0,0.0
68,cover,4541,522,10255,2491,7764,0.0,0.0,0.0
70,favourite songs,4388,441,37,6,31,0.0,0.0,0.0
75,good,4228,508,128,25,103,0.0,0.0,0.0
76,piano,4213,436,0,0,0,0.0,0.0,0.0
79,soundtrack,4098,385,0,0,0,0.0,0.0,0.0
81,soft rock,3987,381,0,0,0,0.0,0.0,0.0


Tags with zero F1: 432 / 658
Tags never predicted: 386


In [ ]:
y_true_check = np.asarray(y_true, dtype=np.int8)
y_pred_check = (y_prob >= 0.15).astype(np.int8)

assert y_true_check.shape == y_pred_check.shape
assert y_true_check.shape[1] == len(selected_tags)
assert Y_train.shape[1] == len(selected_tags)

tp = ((y_pred_check == 1) & (y_true_check == 1)).sum(axis=0)
fp = ((y_pred_check == 1) & (y_true_check == 0)).sum(axis=0)
fn = ((y_pred_check == 0) & (y_true_check == 1)).sum(axis=0)

train_frequency = np.asarray(Y_train.sum(axis=0)).ravel().astype(int)
test_support = y_true_check.sum(axis=0).astype(int)
predicted_count = y_pred_check.sum(axis=0).astype(int)

precision = np.divide(
    tp, tp + fp,
    out=np.zeros(len(selected_tags), dtype=float),
    where=(tp + fp) > 0
)
recall = np.divide(
    tp, tp + fn,
    out=np.zeros(len(selected_tags), dtype=float),
    where=(tp + fn) > 0
)
f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros(len(selected_tags), dtype=float),
    where=(precision + recall) > 0
)

tag_results = pd.DataFrame({
    "tag": list(selected_tags),
    "train_frequency": train_frequency,
    "test_support": test_support,
    "predicted_count": predicted_count,
    "true_positives": tp,
    "false_positives": fp,
    "false_negatives": fn,
    "precision": precision,
    "recall": recall,
    "f1": f1,
})

print("Check summer, great, and favourite:")
display(
    tag_results[tag_results["tag"].isin(
        ["summer", "great", "favourite"]
    )]
)

print("Tags with zero F1:", (tag_results["f1"] == 0).sum())
print("Tags never predicted:", (tag_results["predicted_count"] == 0).sum())

display(
    tag_results.sort_values(
        ["f1", "train_frequency"],
        ascending=[True, False]
    ).head(20)
)

Check summer, great, and favourite:


,tag,train_frequency,test_support,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1
242,favourite,6275,604,0,0,0,604,0.0,0.0,0.0
305,great,3350,314,0,0,0,314,0.0,0.0,0.0
597,summer,3637,391,4,0,4,391,0.0,0.0,0.0


Tags with zero F1: 432
Tags never predicted: 386


,tag,train_frequency,test_support,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1
242,favourite,6275,604,0,0,0,604,0.0,0.0,0.0
240,favorite songs,5981,588,7,0,7,588,0.0,0.0,0.0
392,loved,5297,589,0,0,0,589,0.0,0.0,0.0
329,heard on pandora,4983,568,0,0,0,568,0.0,0.0,0.0
161,cover,4541,522,3,0,3,522,0.0,0.0,0.0
244,favourite songs,4388,441,0,0,0,441,0.0,0.0,0.0
292,good,4228,508,0,0,0,508,0.0,0.0,0.0
466,piano,4213,436,3,0,3,436,0.0,0.0,0.0
584,soundtrack,4098,385,0,0,0,385,0.0,0.0,0.0
575,soft rock,3987,381,11,0,11,381,0.0,0.0,0.0


In [ ]:
tag_results["max_probability"] = y_prob.max(axis=0)
tag_results["mean_probability"] = y_prob.mean(axis=0)

zero_f1_diagnostics = (
    tag_results[tag_results["f1"] == 0]
    .sort_values("max_probability", ascending=False)
)

display(
    zero_f1_diagnostics[
        [
            "tag",
            "train_frequency",
            "test_support",
            "predicted_count",
            "max_probability",
            "mean_probability",
        ]
    ].head(25).round(4)
)

,tag,train_frequency,test_support,predicted_count,max_probability,mean_probability
447,norwegian,1024,167,12,0.6243,0.0069
429,motown,1007,76,11,0.2686,0.0063
151,classical,662,75,3,0.2486,0.0041
172,dark,3147,372,40,0.2402,0.0219
416,melodic hardcore,1003,105,2,0.2215,0.0058
197,dub,664,69,5,0.2179,0.0032
355,instrumental,2077,247,2,0.2165,0.0151
625,trip-hop,2198,205,10,0.2135,0.0170
319,guitar virtuoso,976,37,1,0.2057,0.0063
332,heavy,1753,168,14,0.1991,0.0135


In [ ]:
thresholds = [0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15]

zero_f1_indices = np.where(tag_results["f1"].to_numpy() == 0)[0]

assert tag_results["tag"].tolist() == list(selected_tags)

threshold_rows = []

for tag_idx in zero_f1_indices:
    truth = y_true[:, tag_idx].astype(bool)
    probabilities = y_prob[:, tag_idx]

    for threshold in thresholds:
        pred = probabilities >= threshold

        tp = np.sum(pred & truth)
        fp = np.sum(pred & ~truth)
        fn = np.sum(~pred & truth)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        threshold_rows.append({
            "tag": selected_tags[tag_idx],
            "threshold": threshold,
            "predicted_count": int(pred.sum()),
            "true_positives": int(tp),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

threshold_diagnostics = pd.DataFrame(threshold_rows)

best_zero_f1_results = (
    threshold_diagnostics
    .sort_values(["tag", "f1", "precision"], ascending=[True, False, False])
    .groupby("tag", as_index=False)
    .head(1)
    .sort_values("f1", ascending=False)
)

display(best_zero_f1_results.head(25).round(4))

print(
    "Tags with at least one true positive at a lower threshold:",
    (best_zero_f1_results["true_positives"] > 0).sum()
)

,tag,threshold,predicted_count,true_positives,precision,recall,f1
582,bluegrass,0.020,508,83,0.1634,0.3109,0.2142
2130,old school death metal,0.030,338,39,0.1154,0.5342,0.1898
2054,new york,0.050,263,37,0.1407,0.2357,0.1762
2768,sweden,0.050,133,21,0.1579,0.1981,0.1757
1900,mathcore,0.050,178,21,0.1180,0.2360,0.1573
2124,old school,0.050,408,51,0.1250,0.2116,0.1572
1199,female fronted metal,0.030,641,67,0.1045,0.2888,0.1535
2774,swedish metal,0.030,777,67,0.0862,0.5877,0.1504
1992,motown,0.075,188,18,0.0957,0.2368,0.1364
836,death,0.050,386,35,0.0907,0.2431,0.1321


Tags with at least one true positive at a lower threshold: 429


In [ ]:
assert val_prob.shape == y_val.shape
assert val_prob.shape[1] == len(selected_tags)

thresholds = [0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30]

global_pred = (val_prob >= 0.15).astype(np.int8)
global_micro_f1 = f1_score(
    y_val, global_pred, average="micro", zero_division=0
)

per_tag_rows = []

for j, tag in enumerate(selected_tags):
    best_threshold = 0.15
    best_f1 = -1.0

    for threshold in thresholds:
        pred_j = (val_prob[:, j] >= threshold).astype(np.int8)
        f1_j = f1_score(
            y_val[:, j], pred_j, zero_division=0
        )

        if f1_j > best_f1:
            best_f1 = f1_j
            best_threshold = threshold

    per_tag_rows.append({
        "tag": tag,
        "train_frequency": int(Y_train[:, j].sum()),
        "best_threshold": best_threshold,
        "validation_f1": best_f1,
    })

per_tag_thresholds = pd.DataFrame(per_tag_rows)

threshold_vector = per_tag_thresholds["best_threshold"].to_numpy()
tuned_val_pred = (val_prob >= threshold_vector).astype(np.int8)

tuned_micro_f1 = f1_score(
    y_val, tuned_val_pred, average="micro", zero_division=0
)

tuned_macro_f1 = f1_score(
    y_val, tuned_val_pred, average="macro", zero_division=0
)

global_macro_f1 = f1_score(
    y_val, global_pred, average="macro", zero_division=0
)

print(f"Global threshold 0.15 - validation micro F1: {global_micro_f1:.4f}")
print(f"Per-tag thresholds - validation micro F1: {tuned_micro_f1:.4f}")
print(f"Global threshold 0.15 - validation macro F1: {global_macro_f1:.4f}")
print(f"Per-tag thresholds - validation macro F1: {tuned_macro_f1:.4f}")

print("\nSelected threshold distribution:")
display(
    per_tag_thresholds["best_threshold"]
    .value_counts()
    .sort_index()
    .rename_axis("threshold")
    .to_frame("number_of_tags")
)

print("\nExample tags with the lowest selected thresholds:")
display(
    per_tag_thresholds.sort_values("best_threshold").head(20)
)

Global threshold 0.15 — validation micro F1: 0.2511
Per-tag thresholds — validation micro F1: 0.1496
Global threshold 0.15 — validation macro F1: 0.0682
Per-tag thresholds — validation macro F1: 0.1214

Selected threshold distribution:


,number_of_tags
threshold,
0.010,109
0.020,136
0.030,102
0.050,111
0.075,72
0.100,59
0.150,42
0.200,21
0.300,6



Example tags with the lowest selected thresholds:


,tag,train_frequency,best_threshold,validation_f1
657,yes,693,0.01,0.019298
508,radio,625,0.01,0.025890
503,quiet,894,0.01,0.030608
502,q3,507,0.01,0.016097
270,freakey stylish,529,0.01,0.040541
271,freedom,1012,0.01,0.025018
477,positive,738,0.01,0.030103
283,gdchills90s,529,0.01,0.011364
111,bouncy,657,0.01,0.037736
293,good beat,666,0.01,0.018850


In [ ]:
validation_support = y_val.sum(axis=0)

minimum_support_values = [25, 50, 100, 200]
thresholds = [0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30]

global_pred = (val_prob >= 0.15).astype(np.int8)

results = []

for minimum_support in minimum_support_values:
    thresholds_for_tags = np.full(len(selected_tags), 0.15)

    for j in range(len(selected_tags)):
        if validation_support[j] < minimum_support:
            continue

        best_threshold = 0.15
        best_f1 = -1.0

        for threshold in thresholds:
            pred_j = (val_prob[:, j] >= threshold).astype(np.int8)
            score = f1_score(
                y_val[:, j], pred_j, zero_division=0
            )

            if score > best_f1:
                best_f1 = score
                best_threshold = threshold

        thresholds_for_tags[j] = best_threshold

    tuned_pred = (
        val_prob >= thresholds_for_tags[np.newaxis, :]
    ).astype(np.int8)

    results.append({
        "minimum_validation_support": minimum_support,
        "tags_tuned": int(
            (validation_support >= minimum_support).sum()
        ),
        "micro_f1": f1_score(
            y_val, tuned_pred, average="micro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_val, tuned_pred, average="macro", zero_division=0
        ),
        "mean_predicted_tags_per_track": tuned_pred.sum(axis=1).mean(),
    })

display(pd.DataFrame(results).round(4))

,minimum_validation_support,tags_tuned,micro_f1,macro_f1,mean_predicted_tags_per_track
0,25,658,0.1496,0.1214,41.5307
1,50,629,0.1513,0.1197,40.7530
2,100,331,0.1794,0.0978,30.1843
3,200,165,0.2056,0.0823,22.3563


In [ ]:
minimum_support = 200
validation_support = y_val.sum(axis=0)

thresholds_for_tags = np.full(len(selected_tags), 0.15)

for j in range(len(selected_tags)):
    if validation_support[j] < minimum_support:
        continue

    best_threshold = 0.15
    best_f1 = -1.0

    for threshold in [0.01, 0.02, 0.03, 0.05, 0.075,
                      0.10, 0.15, 0.20, 0.30]:
        pred_j = (val_prob[:, j] >= threshold).astype(np.int8)
        score = f1_score(y_val[:, j], pred_j, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    thresholds_for_tags[j] = best_threshold

threshold_summary = (
    pd.Series(thresholds_for_tags)
    .value_counts()
    .sort_index()
    .rename_axis("threshold")
    .reset_index(name="number_of_tags")
)

display(threshold_summary)

,threshold,number_of_tags
0,0.020,4
1,0.030,17
2,0.050,33
3,0.075,34
4,0.100,37
5,0.150,518
6,0.200,12
7,0.300,3


In [ ]:
metadata = pd.read_csv(
    PROCESSED_DIR / "track_metadata_lookup.csv"
)

metadata["track_id"] = metadata["track_id"].astype(str)
labeled_ids = pd.Series(train_ids_labeled).astype(str)

labeled_metadata = metadata[
    metadata["track_id"].isin(labeled_ids)
].copy()

print("Labeled training tracks:", len(labeled_ids))
print("Tracks with metadata:", len(labeled_metadata))
print("Missing titles:", labeled_metadata["title"].isna().sum())
print("Missing artists:", labeled_metadata["artist_name"].isna().sum())
print("Missing years:", labeled_metadata["year"].isna().sum())

display(
    labeled_metadata[
        ["track_id", "title", "artist_name", "year"]
    ].head(10)
)

Labeled training tracks: 140348
Tracks with metadata: 140348
Missing titles: 1
Missing artists: 0
Missing years: 0


,track_id,title,artist_name,year
27144,TRAAABD128F429CF47,Soul Deep,The Box Tops,1969
27145,TRAAAED128E0783FAB,It's About Time,Jamie Cullum,0
27146,TRAAAEF128F4273421,Something Girls,Adam Ant,1982
27148,TRAAAFD128F92F423A,Face the Ashes,Gob,2007
27149,TRAAAGF12903CEC202,Små Ord,Halvdan Sivertsen,2005
27151,TRAAAHZ128E0799171,The One And Only (Edited),Snoop Dogg,0
27154,TRAAARJ128F9320760,Pink World,Planet P Project,1984
27156,TRAAAZF12903CCCF6B,Break My Stride,Matthew Wilder,1983
27157,TRAABEV12903CC53A4,Blood In Face,Suicide Commando,2000
27160,TRAABIG128F9356C56,Walk the Walk,Poe,2000


In [ ]:
metadata = pd.read_csv(
    PROCESSED_DIR / "track_metadata_lookup.csv",
    dtype={"track_id": str}
)

metadata = metadata.drop_duplicates(subset="track_id").copy()

metadata["year"] = pd.to_numeric(
    metadata["year"], errors="coerce"
).replace(0, np.nan)

train_ids_str = pd.Series(train_ids_labeled).astype(str)

catalog = pd.DataFrame({"track_id": train_ids_str})
catalog = catalog.merge(
    metadata[["track_id", "title", "artist_name", "year"]],
    on="track_id",
    how="left",
    validate="one_to_one"
)

valid_title = catalog["title"].notna() & (
    catalog["title"].str.strip() != ""
)

print("Catalog rows:", len(catalog))
print("Rows missing title:", (~valid_title).sum())
print("Rows missing artist:", catalog["artist_name"].isna().sum())
print("Unknown release years:", catalog["year"].isna().sum())

display(catalog.head(10))

Catalog rows: 140348
Rows missing title: 1
Rows missing artist: 0
Unknown release years: 21899


,track_id,title,artist_name,year
0,TRAAABD128F429CF47,Soul Deep,The Box Tops,1969.0
1,TRAAAED128E0783FAB,It's About Time,Jamie Cullum,NaN
2,TRAAAEF128F4273421,Something Girls,Adam Ant,1982.0
3,TRAAAFD128F92F423A,Face the Ashes,Gob,2007.0
4,TRAAAGF12903CEC202,Små Ord,Halvdan Sivertsen,2005.0
5,TRAAAHZ128E0799171,The One And Only (Edited),Snoop Dogg,NaN
6,TRAAARJ128F9320760,Pink World,Planet P Project,1984.0
7,TRAAAZF12903CCCF6B,Break My Stride,Matthew Wilder,1983.0
8,TRAABEV12903CC53A4,Blood In Face,Suicide Commando,2000.0
9,TRAABIG128F9356C56,Walk the Walk,Poe,2000.0


In [ ]:
assert len(catalog) == X_train_tfidf.shape[0]
assert catalog["track_id"].is_unique

valid_title = (
    catalog["title"].notna()
    & catalog["title"].str.strip().ne("")
)

catalog_rec = (
    catalog.loc[valid_title]
    .reset_index(drop=True)
    .copy()
)

X_rec = X_train_tfidf[valid_title.to_numpy()].tocsr()

assert len(catalog_rec) == X_rec.shape[0]

print("Recommendation catalog:", len(catalog_rec))
print("Feature matrix:", X_rec.shape)

Recommendation catalog: 140347
Feature matrix: (140347, 5000)


<b>Recommendation function creation and initial testing</b>

In [ ]:
def recommend_similar_tracks(
    track_id,
    catalog=catalog_rec,
    features=X_rec,
    n=10,
    exclude_track_ids=None,
):
    matches = np.flatnonzero(
        catalog["track_id"].to_numpy() == str(track_id)
    )

    if len(matches) == 0:
        raise ValueError(
            f"Track ID {track_id!r} is not in the catalog."
        )

    query_idx = matches[0]

    scores = cosine_similarity(
        features[query_idx],
        features
    ).ravel()

    scores[query_idx] = -1

    excluded = set(map(str, exclude_track_ids or []))
    excluded.add(str(track_id))

    excluded_mask = catalog["track_id"].isin(excluded).to_numpy()
    scores[excluded_mask] = -1

    candidate_idx = np.flatnonzero(scores >= 0)
    top_idx = candidate_idx[
        np.argsort(scores[candidate_idx])[::-1][:n]
    ]

    results = catalog.iloc[top_idx][
        ["track_id", "title", "artist_name", "year"]
    ].copy()

    results["lyric_similarity"] = scores[top_idx]
    return results.reset_index(drop=True)

In [ ]:
example_track = catalog_rec.loc[
    catalog_rec["title"].eq("Break My Stride")
    & catalog_rec["artist_name"].eq("Matthew Wilder")
]

if example_track.empty:
    print("Example track not found.")
else:
    example_id = example_track.iloc[0]["track_id"]

    display(
        recommend_similar_tracks(
            track_id=example_id,
            n=10,
        ).round({"lyric_similarity": 3})
    )

,track_id,title,artist_name,year,lyric_similarity
0,TRNUPYQ128F92F6556,Break My Stride,Blue Lagoon,NaN,0.807
1,TRASFWC128F14A0CDB,Break My Stride (Radio Edit),Bluelagoon,NaN,0.789
2,TRMMSWR128F93263D5,Can't Slow Down,Lionel Richie,1983.0,0.368
3,TRHBGQF128F424DE92,Let It Ride,Shed Seven,1998.0,0.336
4,TRJCPIL128F428AAD8,Boys Nite Out,Krokus,1984.0,0.326
5,TRBSHRP128F42A9F37,Movin' On,Backstreet Boys,2005.0,0.319
6,TRXFEHB12903CEA4F6,Movin' On,Bad Company,1974.0,0.319
7,TRNBISB128F4292F46,You Keep On Lovin' Me,Sherrié Austin,1997.0,0.315
8,TRQVWMD12903CAEF6A,I Am,Mary J. Blige,2009.0,0.312
9,TRHWTLP128F428A858,Getting It Off My Chest (Album Version),Death By Stereo,2001.0,0.312


<b>Exporting the model</b>

In [ ]:
import joblib
import pickle
from scipy.sparse import save_npz

model.save("lyric_tag_model.keras")

joblib.dump(tfidf, "tfidf_transformer.joblib")

with open("vocab_tags.pkl", "wb") as f:
    pickle.dump({
        "vocab": vocab, 
        "selected_tags": selected_tags
    }, f)

catalog_rec.to_pickle("catalog_rec.pkl")
save_npz("X_rec.npz", X_rec)